# EDA — Sentimen Ulasan Mobile JKN (2025–2026)

Analisis eksploratif atas **97.566 ulasan** Mobile JKN (Google Play) yang sudah
diklasifikasi sentimennya oleh pipeline.

**Isi:**
1. Muat data (`data/sentiment_daily.csv`, `data/backfill/monthly.csv`)
2. Ringkasan & statistik dasar
3. Tren harian + rata-rata bergerak 7 hari
4. Pola bulanan (musiman)
5. Query SQL dengan DuckDB (agregasi, hari terburuk, pola hari)
6. Kesimpulan

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

ROOT = Path.cwd()
DATA = ROOT / "data" if (ROOT / "data").exists() else ROOT.parent / "data"
print("Folder data:", DATA.resolve())

In [ ]:
daily = pd.read_csv(DATA / "sentiment_daily.csv", parse_dates=["date"])
monthly = pd.read_csv(DATA / "backfill" / "monthly.csv")
daily.head()

## 2. Ringkasan & statistik dasar

In [ ]:
print("Rentang :", daily["date"].min().date(), "->", daily["date"].max().date())
print("Hari    :", len(daily))
print("Ulasan  :", int(daily["total"].sum()))
print("Rata2 rasio negatif:", round(daily["neg_ratio"].mean(), 4))
daily[["total", "pos_ratio", "neg_ratio"]].describe().round(3)

## 3. Tren harian + rata-rata bergerak 7 hari

Garis tipis = nilai harian (berisik). Garis tebal = rata-rata 7 hari (tren sebenarnya).

In [ ]:
d = daily.set_index("date").sort_index()
d["neg_ma7"] = d["neg_ratio"].rolling(7, min_periods=1).mean()

ax = d["neg_ratio"].plot(alpha=0.30, label="negatif harian")
d["neg_ma7"].plot(ax=ax, color="crimson", linewidth=2, label="rata-rata 7 hari")
ax.set_title("Rasio ulasan negatif — Mobile JKN")
ax.set_ylabel("rasio negatif")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Pola bulanan (musiman)

In [ ]:
m = monthly.set_index("bulan").sort_index()
ax = m[["pos_ratio", "neg_ratio"]].plot(kind="bar")
ax.set_title("Sentimen per bulan (2025–2026)")
ax.set_ylabel("rasio")
ax.set_xlabel("bulan")
plt.tight_layout()
plt.show()

print("5 bulan paling negatif:")
print(m["neg_ratio"].sort_values(ascending=False).head(5))

## 5. Query SQL dengan DuckDB

DuckDB bisa langsung membaca DataFrame seperti tabel SQL.

In [ ]:
con = duckdb.connect()
con.register("daily", daily)

### 5a. Agregat per bulan

In [ ]:
con.sql("""
    SELECT strftime(date, '%Y-%m')            AS bulan,
           SUM(total)                          AS ulasan,
           ROUND(AVG(neg_ratio), 4)            AS rata_negatif,
           ROUND(MAX(neg_ratio), 4)            AS negatif_tertinggi
    FROM daily
    GROUP BY 1
    ORDER BY 1
""").df()

### 5b. 10 hari dengan rasio negatif tertinggi (min. 100 ulasan)

In [ ]:
con.sql("""
    SELECT date,
           total,
           ROUND(neg_ratio * 100, 1) AS neg_persen
    FROM daily
    WHERE total >= 100
    ORDER BY neg_ratio DESC
    LIMIT 10
""").df()

### 5c. Pola per hari dalam seminggu

0 = Minggu, 1 = Senin, ..., 6 = Sabtu.

In [ ]:
nama_hari = {0: "Min", 1: "Sen", 2: "Sel", 3: "Rab", 4: "Kam", 5: "Jum", 6: "Sab"}

hasil = con.sql("""
    SELECT CAST(strftime(date, '%w') AS INTEGER) AS hari_ke,
           ROUND(AVG(neg_ratio), 4)              AS rata_negatif,
           SUM(total)                            AS ulasan
    FROM daily
    GROUP BY 1
    ORDER BY 1
""").df()
hasil["hari"] = hasil["hari_ke"].map(nama_hari)
hasil[["hari", "rata_negatif", "ulasan"]]

## 6. Kesimpulan

- Sentimen Mobile JKN secara umum **positif** (rasio positif sering > 45%),
  tetapi ada bulan dengan lonjakan negatif yang mencolok.
- Lonjakan negatif biasanya terkait **masalah layanan** (login/OTP, verifikasi
  wajah, antrean daring) — lihat notebook lanjutan untuk kata kunci.
- Pola ini yang dipakai pipeline untuk **deteksi anomali** dan **alert otomatis**:
  ketika rasio negatif melampaui baseline, sistem membuat GitHub Issue + narasi AI.

**Langkah lanjut:**
- Analisis kata/kata kunci dari `data/backfill/raw_reviews.jsonl.gz`.
- Uji model ML (SVM/TF-IDF) sebagai pengganti lexicon di `pipeline/classify.py`.
- Sambungkan agregat ke Looker Studio (lihat `docs/looker_studio.md`).